# Inspect `openai/privacy-filter` Architecture

Dieses Notebook lädt das Modell und gibt die Architektur/Schichten aus.

In [7]:
from transformers import AutoConfig, AutoModelForTokenClassification

model_id = "openai/privacy-filter"
print(f"Model: {model_id}")

Model: openai/privacy-filter


In [8]:
config = AutoConfig.from_pretrained(model_id)
print("=== Config ===")
print(config)
print("\nModel type:", config.model_type)
print("Hidden size:", getattr(config, "hidden_size", "n/a"))
print("Num hidden layers:", getattr(config, "num_hidden_layers", "n/a"))
print("Num attention heads:", getattr(config, "num_attention_heads", "n/a"))
print("Num labels:", getattr(config, "num_labels", "n/a"))

=== Config ===
OpenAIPrivacyFilterConfig {
  "architectures": [
    "OpenAIPrivacyFilterForTokenClassification"
  ],
  "attention_bias": true,
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "classifier_dropout": 0.0,
  "default_n_ctx": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 199999,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 640,
  "id2label": {
    "0": "O",
    "1": "B-account_number",
    "2": "I-account_number",
    "3": "E-account_number",
    "4": "S-account_number",
    "5": "B-private_address",
    "6": "I-private_address",
    "7": "E-private_address",
    "8": "S-private_address",
    "9": "B-private_date",
    "10": "I-private_date",
    "11": "E-private_date",
    "12": "S-private_date",
    "13": "B-private_email",
    "14": "I-private_email",
    "15": "E-private_email",
    "16": "S-private_email",
    "17": "B-private_person",
    "18": "I-private_person",
    "19": "E-private_person",
    "20": "S-private_person",
    "21": "B-private_ph

In [9]:
model = AutoModelForTokenClassification.from_pretrained(model_id)
print("=== Model Class ===")
print(model.__class__.__name__)
print("\n=== Full Architecture ===")
print(model)

Loading weights: 100%|██████████| 140/140 [00:00<00:00, 7165.11it/s]

=== Model Class ===
OpenAIPrivacyFilterForTokenClassification

=== Full Architecture ===
OpenAIPrivacyFilterForTokenClassification(
  (model): OpenAIPrivacyFilterModel(
    (embed_tokens): Embedding(200064, 640, padding_idx=199999)
    (layers): ModuleList(
      (0-7): 8 x OpenAIPrivacyFilterEncoderLayer(
        (self_attn): OpenAIPrivacyFilterAttention(
          (q_proj): Linear(in_features=640, out_features=896, bias=True)
          (k_proj): Linear(in_features=640, out_features=128, bias=True)
          (v_proj): Linear(in_features=640, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=640, bias=True)
        )
        (mlp): OpenAIPrivacyFilterMLP(
          (router): OpenAIPrivacyFilterTopKRouter()
          (experts): OpenAIPrivacyFilterExperts()
        )
        (input_layernorm): OpenAIPrivacyFilterRMSNorm((640,), eps=1e-05)
        (post_attention_layernorm): OpenAIPrivacyFilterRMSNorm((640,), eps=1e-05)
      )
    )
    (norm): OpenAIP

In [10]:
print("=== Top-Level Modules ===")
for name, module in model.named_children():
    print(f"{name}: {module.__class__.__name__}")

encoder = None
for candidate in ["bert", "roberta", "distilbert", "deberta", "electra", "encoder", "backbone"]:
    if hasattr(model, candidate):
        encoder = getattr(model, candidate)
        print(f"\nBackbone attribute found: {candidate} ({encoder.__class__.__name__})")
        break

if encoder is not None:
    if hasattr(encoder, "encoder") and hasattr(encoder.encoder, "layer"):
        layers = encoder.encoder.layer
        print(f"Transformer blocks: {len(layers)}")
    elif hasattr(encoder, "transformer") and hasattr(encoder.transformer, "layer"):
        layers = encoder.transformer.layer
        print(f"Transformer blocks: {len(layers)}")
    elif hasattr(encoder, "layer"):
        layers = encoder.layer
        print(f"Transformer blocks: {len(layers)}")
    else:
        print("Transformer blocks konnten nicht automatisch bestimmt werden.")

=== Top-Level Modules ===
model: OpenAIPrivacyFilterModel
dropout: Dropout
score: Linear


In [11]:
import torch
from collections import Counter

print("=== Precision / DType Analysis ===")
param_dtypes = Counter(str(p.dtype) for p in model.parameters())
buffer_dtypes = Counter(str(b.dtype) for b in model.buffers())

print("\nParameter dtypes:")
for dtype, count in param_dtypes.most_common():
    print(f"  {dtype}: {count} tensors")

print("\nBuffer dtypes:")
if buffer_dtypes:
    for dtype, count in buffer_dtypes.most_common():
        print(f"  {dtype}: {count} tensors")
else:
    print("  (no buffers)")

main_dtype = next(model.parameters()).dtype
print(f"\nMain parameter dtype: {main_dtype}")

print("\nTorch backend support:")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"  bf16 supported: {torch.cuda.is_bf16_supported()}")
    print("  fp16 is generally supported on CUDA GPUs.")
else:
    print("  Running on CPU; fp16/bf16 speedups are usually GPU-specific.")


=== Precision / DType Analysis ===

Parameter dtypes:
  torch.bfloat16: 132 tensors
  torch.float32: 8 tensors

Buffer dtypes:
  torch.float32: 2 tensors

Main parameter dtype: torch.bfloat16

Torch backend support:
  CUDA available: False
  Running on CPU; fp16/bf16 speedups are usually GPU-specific.


In [12]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== Parameter Summary ===")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\n=== First 40 named parameters ===")
for i, (name, param) in enumerate(model.named_parameters()):
    if i >= 80:
        break
    print(f"{name:80s} {tuple(param.shape)}")

=== Parameter Summary ===
Total parameters:     1,399,486,865
Trainable parameters: 1,399,486,865

=== First 40 named parameters ===
model.embed_tokens.weight                                                        (200064, 640)
model.layers.0.self_attn.sinks                                                   (14,)
model.layers.0.self_attn.q_proj.weight                                           (896, 640)
model.layers.0.self_attn.q_proj.bias                                             (896,)
model.layers.0.self_attn.k_proj.weight                                           (128, 640)
model.layers.0.self_attn.k_proj.bias                                             (128,)
model.layers.0.self_attn.v_proj.weight                                           (128, 640)
model.layers.0.self_attn.v_proj.bias                                             (128,)
model.layers.0.self_attn.o_proj.weight                                           (640, 896)
model.layers.0.self_attn.o_proj.bias                 

In [ ]:
# === INT8 Quantization + Eval + Optional Hub Push ===
# Voraussetzungen:
# 1) model und tokenizer sind bereits geladen (aus den Zellen oben)
# 2) pip install datasets tqdm huggingface_hub

import ast
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from tqdm import tqdm

# -------------------- Konfiguration --------------------
DATASET_NAME = "ai4privacy/open-pii-masking-500k-ai4privacy"
SPLIT = "validation"
LANGUAGE = "de"
EVAL_SAMPLES = 1500
SEED = 42
BATCH_SIZE = 8

# Lokaler Ausgabeordner fuer quantisiertes Modell + Metriken
OUTPUT_DIR = Path("privacy_filter_int8_notebook")

# Optionaler Push zu HF Hub
PUSH_TO_HUB = False
REPO_ID = "<dein-user>/<privacy-filter-int8>"
HF_TOKEN = None  # oder String setzen
PRIVATE_REPO = True


@dataclass
class TokenMetrics:
    tp: int = 0
    fp: int = 0
    fn: int = 0
    tn: int = 0

    @property
    def precision(self) -> float:
        denom = self.tp + self.fp
        return self.tp / denom if denom else 0.0

    @property
    def recall(self) -> float:
        denom = self.tp + self.fn
        return self.tp / denom if denom else 0.0

    @property
    def f1(self) -> float:
        p = self.precision
        r = self.recall
        return (2 * p * r / (p + r)) if (p + r) else 0.0

    def update(self, y_true: list[bool], y_pred: list[bool]) -> None:
        for truth, pred in zip(y_true, y_pred):
            if pred and truth:
                self.tp += 1
            elif pred and not truth:
                self.fp += 1
            elif truth and not pred:
                self.fn += 1
            else:
                self.tn += 1


@dataclass
class SpanMetrics:
    tp: int = 0
    fp: int = 0
    fn: int = 0

    @property
    def f1(self) -> float:
        denom = 2 * self.tp + self.fp + self.fn
        return (2 * self.tp) / denom if denom else 0.0

    def update(self, truth_spans: set[tuple[int, int]], pred_spans: set[tuple[int, int]]) -> None:
        self.tp += len(truth_spans & pred_spans)
        self.fp += len(pred_spans - truth_spans)
        self.fn += len(truth_spans - pred_spans)


def load_validation_dataset(dataset_name: str, split_name: str):
    try:
        return load_dataset(dataset_name, split=split_name)
    except Exception:
        train_ds = load_dataset(dataset_name, split="train")
        if "set" not in train_ds.column_names:
            raise
        filtered = train_ds.filter(lambda row: str(row.get("set", "")).lower() == split_name.lower())
        if len(filtered) == 0:
            raise ValueError(f"Split '{split_name}' not found in dataset '{dataset_name}'.")
        return filtered


def parse_span_labels(raw_span_labels) -> list[tuple[int, int]]:
    if raw_span_labels is None:
        return []
    if isinstance(raw_span_labels, str):
        raw_span_labels = ast.literal_eval(raw_span_labels)

    spans: list[tuple[int, int]] = []
    for item in raw_span_labels:
        if isinstance(item, dict):
            start, end = item.get("start"), item.get("end")
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            start, end = item[0], item[1]
        else:
            continue
        if start is None or end is None:
            continue
        start_i, end_i = int(start), int(end)
        if end_i > start_i:
            spans.append((start_i, end_i))
    return merge_spans(spans)


def merge_spans(spans: Iterable[tuple[int, int]]) -> list[tuple[int, int]]:
    ordered = sorted(spans, key=lambda x: (x[0], x[1]))
    if not ordered:
        return []
    merged: list[list[int]] = [[ordered[0][0], ordered[0][1]]]
    for start, end in ordered[1:]:
        last = merged[-1]
        if start <= last[1]:
            last[1] = max(last[1], end)
        else:
            merged.append([start, end])
    return [(start, end) for start, end in merged]


def extract_token_spans(text: str) -> list[tuple[int, int]]:
    return [(match.start(), match.end()) for match in re.finditer(r"\S+", text)]


def mark_tokens_as_pii(token_spans: list[tuple[int, int]], pii_spans: list[tuple[int, int]]) -> list[bool]:
    if not pii_spans:
        return [False] * len(token_spans)
    marks = [False] * len(token_spans)
    span_index = 0
    ordered_spans = sorted(pii_spans, key=lambda x: (x[0], x[1]))
    for idx, (tok_start, tok_end) in enumerate(token_spans):
        while span_index < len(ordered_spans) and ordered_spans[span_index][1] <= tok_start:
            span_index += 1
        check_index = span_index
        while check_index < len(ordered_spans) and ordered_spans[check_index][0] < tok_end:
            span_start, span_end = ordered_spans[check_index]
            if span_end > tok_start and span_start < tok_end:
                marks[idx] = True
                break
            check_index += 1
    return marks


def label_is_pii(label: str) -> bool:
    if not label:
        return False
    return str(label).upper() != "O"


def predict_spans_for_batch(texts: list[str], tokenizer, model) -> list[list[tuple[int, int]]]:
    encoded = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        return_offsets_mapping=True,
    )
    offset_mapping = encoded.pop("offset_mapping")

    with torch.inference_mode():
        logits = model(**encoded).logits

    predicted_ids = logits.argmax(dim=-1).cpu()
    input_ids = encoded["input_ids"]

    batch_spans: list[list[tuple[int, int]]] = []
    for sample_idx in range(len(texts)):
        spans: list[tuple[int, int]] = []
        token_ids = input_ids[sample_idx]
        token_offsets = offset_mapping[sample_idx]
        sample_pred_ids = predicted_ids[sample_idx]

        for token_id, class_id, (start, end) in zip(token_ids, sample_pred_ids, token_offsets):
            token_id_int = int(token_id.item())
            if token_id_int in tokenizer.all_special_ids:
                continue

            start_i = int(start.item())
            end_i = int(end.item())
            if end_i <= start_i:
                continue

            label = model.config.id2label[int(class_id.item())]
            if label_is_pii(label):
                spans.append((start_i, end_i))

        batch_spans.append(merge_spans(spans))
    return batch_spans


def evaluate_quantized_model(dataset, tokenizer, model, batch_size: int = 8) -> dict:
    token_metrics = TokenMetrics()
    span_metrics = SpanMetrics()

    rows_batch: list[dict] = []
    texts_batch: list[str] = []
    gt_spans_batch: list[list[tuple[int, int]]] = []

    def flush_batch() -> None:
        if not texts_batch:
            return

        pred_spans_batch = predict_spans_for_batch(texts_batch, tokenizer, model)
        for text, gt_spans, pred_spans in zip(texts_batch, gt_spans_batch, pred_spans_batch):
            token_spans = extract_token_spans(text)
            y_true = mark_tokens_as_pii(token_spans, gt_spans)
            y_pred = mark_tokens_as_pii(token_spans, pred_spans)
            token_metrics.update(y_true, y_pred)
            span_metrics.update(set(gt_spans), set(pred_spans))

        rows_batch.clear()
        texts_batch.clear()
        gt_spans_batch.clear()

    for row in tqdm(dataset, total=len(dataset), desc="INT8 eval", unit="sample", dynamic_ncols=True):
        text = row.get("source_text") or ""
        if not text:
            continue
        gt_spans = parse_span_labels(row.get("privacy_mask"))
        rows_batch.append(row)
        texts_batch.append(text)
        gt_spans_batch.append(gt_spans)
        if len(texts_batch) >= max(1, batch_size):
            flush_batch()

    flush_batch()

    return {
        "precision": token_metrics.precision,
        "recall": token_metrics.recall,
        "f1_tokens": token_metrics.f1,
        "f1_spans": span_metrics.f1,
    }


# -------------------- Ablauf --------------------
print("Loading dataset...")
dataset = load_validation_dataset(DATASET_NAME, SPLIT)
if "language" in dataset.column_names:
    dataset = dataset.filter(lambda row: str(row.get("language", "")).lower() == LANGUAGE.lower())

if len(dataset) == 0:
    raise ValueError("Keine Samples nach language-Filter gefunden.")

sample_count = min(EVAL_SAMPLES, len(dataset))
dataset = dataset.shuffle(seed=SEED).select(range(sample_count))
print(f"Eval samples: {sample_count}")

print("Quantizing model to INT8 (dynamic quantization on Linear layers)...")
quantized_model = torch.quantization.quantize_dynamic(
    model.cpu(),
    {torch.nn.Linear},
    dtype=torch.qint8,
)
quantized_model.eval()

metrics = evaluate_quantized_model(dataset, tokenizer, quantized_model, batch_size=BATCH_SIZE)

print("\n=== INT8 Validation Metrics (de, 1500 samples) ===")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1 tokens: {metrics['f1_tokens']:.4f}")
print(f"F1 spans:  {metrics['f1_spans']:.4f}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quantized_model.save_pretrained(OUTPUT_DIR, safe_serialization=False)
tokenizer.save_pretrained(OUTPUT_DIR)
(OUTPUT_DIR / "int8_eval_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"\nSaved quantized model + metrics to: {OUTPUT_DIR}")

if PUSH_TO_HUB:
    if REPO_ID.startswith("<"):
        raise ValueError("Bitte REPO_ID auf ein echtes HF Repo setzen.")
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=REPO_ID, private=PRIVATE_REPO, exist_ok=True)
    api.upload_folder(
        repo_id=REPO_ID,
        folder_path=str(OUTPUT_DIR),
        commit_message="Add INT8 quantized model + eval metrics (de validation 1500)",
    )
    print(f"Pushed to: https://huggingface.co/{REPO_ID}")
else:
    print("Hub push skipped (PUSH_TO_HUB=False).")
